# Lecture 17: 3D Vision

*Notebook for UMich EECS 498 (Justin Johnson), Lecture 17 "3D Vision" (Fall 2019 slides).*

Last time we predicted **2D** shapes of objects (Mask R-CNN). Today: **3D** shapes. This lecture focuses on two problems: **predicting 3D shapes from a single image**, and **processing 3D input data**.

**What you'll learn**
1. The five main **3D shape representations**: depth maps, voxel grids, implicit surfaces, point clouds, and triangle meshes, each with its pros and cons.
2. How neural networks **process** and **predict** each one: 3D convolution, PointNet, occupancy networks, graph convolution.
3. **Loss functions** for 3D shapes, especially the **Chamfer distance**.
4. **Metrics** for comparing shapes (IoU, Chamfer, F1), camera coordinate systems, datasets, and **Mesh R-CNN**.

Every idea has a small working example: we'll train a PointNet, an occupancy network, and fit point clouds and meshes with Chamfer loss, all from scratch.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def softmax(scores, axis=-1):
    e = np.exp(scores - scores.max(axis=axis, keepdims=True))
    return e / e.sum(axis=axis, keepdims=True)

**Many more topics in 3D vision** go beyond this lecture: computing correspondences, multi-view stereo, structure from motion, SLAM, view synthesis, differentiable graphics, 3D sensors… and many **non-deep-learning** methods are alive and well in 3D!

| Representation | What it stores | Analogy |
|---|---|---|
| **Depth map** | distance to the scene at each pixel | a 2D image of depths |
| **Voxel grid** | occupancy of each cell in a $V\times V\times V$ grid | a 3D segmentation mask |
| **Implicit surface** | a function: "is point $(x, y, z)$ inside?" | a classifier over space |
| **Point cloud** | a set of $P$ points on the surface | a scatter of samples |
| **Triangle mesh** | vertices + triangular faces | what computer graphics uses |

## 1. Depth maps

For each pixel, a **depth map** gives the **distance from the camera** to the object at that pixel. An RGB image plus a depth image is an **RGB-D image** ("2.5D"): it only describes the visible surface. Some 3D sensors (e.g., Microsoft Kinect) record depth directly.

**Predicting depth maps:** a fully convolutional network (Lecture 16) takes the $3\times H\times W$ image and outputs a $1\times H\times W$ depth map, trained with a per-pixel L2 loss.

### Problem: scale / depth ambiguity
**A small, close object looks exactly the same as a larger, farther-away object.** With the pinhole camera from Lecture 1, image position $= f\cdot X/Z$:

In [ ]:
def project(X, Z, f=1.0):
    return f * X / Z

small_close = project(X=1.0, Z=2.0)      # a 1 m object, 2 m away
large_far = project(X=3.0, Z=6.0)        # a 3 m object, 6 m away
print(f"image size of the small, close object: {small_close}")
print(f"image size of the large, far object:   {large_far}   <- identical")

So **absolute scale and depth are ambiguous from a single image**. Eigen et al. (2014) used a **scale-invariant loss**. With $d_i = \log y_i - \log y_i^*$ (the log-difference between predicted and true depth at pixel $i$):

$$D(y, y^*) = \frac{1}{n}\sum_i d_i^2 - \frac{\lambda}{n^2}\left(\sum_i d_i\right)^2$$

With $\lambda = 1$, multiplying the whole predicted depth map by any constant gives **zero** extra loss: that shifts every $d_i$ by the same amount, which the second term cancels. Only **relative** depth matters.

In [ ]:
def scale_invariant_loss(pred, true, lam=1.0):
    d = np.log(pred) - np.log(true)
    n = d.size
    return np.sum(d ** 2) / n - lam * np.sum(d) ** 2 / n ** 2

rng = np.random.default_rng(0)
true_depth = rng.uniform(1, 10, (8, 8))
for name, pred in [("perfect", true_depth), ("everything 2x too far", 2 * true_depth),
                   ("right scale, but noisy", true_depth * np.exp(rng.normal(0, 0.2, (8, 8))))]:
    l2 = np.mean((pred - true_depth) ** 2)
    print(f"{name:24s}: L2 loss {l2:7.2f},  scale-invariant loss {abs(scale_invariant_loss(pred, true_depth)):.4f}")

The scale-invariant loss doesn't punish "everything 2× too far" at all, but it does punish wrong **shape**.

### Surface normals
For each pixel, a **surface normal** is the unit vector perpendicular to the object's surface there ($3\times H\times W$). Normals can be computed from a depth map (they're perpendicular to the surface's slope), and a network can predict them directly, with a per-pixel **cosine** loss: $1 - \frac{x\cdot y}{|x||y|}$ (since $x\cdot y = |x||y|\cos\theta$).

In [ ]:
yy, xx = np.mgrid[-1:1:64j, -1:1:64j]
depth = 3.0 - np.sqrt(np.clip(1 - xx ** 2 - yy ** 2, 0, None))       # a sphere bulging toward the camera
dz_dy, dz_dx = np.gradient(depth, yy[:, 0], xx[0])
normals = np.stack([dz_dx, dz_dy, -np.ones_like(depth)], axis=-1)    # perpendicular to the surface's slope
normals /= np.linalg.norm(normals, axis=-1, keepdims=True)

def cosine_loss(pred, true):
    return np.mean(1 - np.sum(pred * true, axis=-1) / (np.linalg.norm(pred, axis=-1) * np.linalg.norm(true, axis=-1)))

print(f"cosine loss of the normals against themselves: {abs(cosine_loss(normals, normals)):.3f}")
print(f"cosine loss against 'everything faces the camera': {cosine_loss(np.broadcast_to([0, 0, -1.0], normals.shape), normals):.3f}")
fig, axes = plt.subplots(1, 2, figsize=(7, 3))
axes[0].imshow(depth, cmap="viridis"); axes[0].set_title("depth map")
axes[1].imshow((normals + 1) / 2); axes[1].set_title("surface normals (as RGB)")
for ax in axes: ax.axis("off")
plt.show()

## 2. Voxel grids

Represent a shape with a $V\times V\times V$ grid of **occupancies** (1 = inside the object). Just like segmentation masks in Mask R-CNN, but in 3D!
* (+) Conceptually simple: just a 3D grid!
* (−) Need high spatial resolution to capture fine structures, and scaling to high resolutions is nontrivial.

In [ ]:
def voxelize(inside_function, V):
    # Occupancy grid: is the center of each voxel inside the shape?
    centers = (np.arange(V) + 0.5) / V * 2 - 1                      # voxel centers in [-1, 1]
    x, y, z = np.meshgrid(centers, centers, centers, indexing="ij")
    return inside_function(x, y, z)

sphere = lambda x, y, z: x ** 2 + y ** 2 + z ** 2 < 0.8 ** 2
for V in [8, 16, 32]:
    volume = voxelize(sphere, V).mean() * 8                          # fraction occupied x volume of the [-1,1]^3 box
    print(f"{V:2d}^3 voxels: estimated sphere volume {volume:.3f}  (true {4 / 3 * np.pi * 0.8 ** 3:.3f})")

### Voxel problems: memory usage
Storing a $V^3$ grid of 32-bit numbers takes $4V^3$ bytes, so memory grows with the **cube** of the resolution:

In [ ]:
for V in [32, 128, 256, 512, 1024]:
    print(f"{V:5d}^3 voxel grid: {4 * V ** 3 / 2 ** 20:8.0f} MB")

**Storing a $1024^3$ voxel grid takes 4 GB of memory!** Ways to scale up: **oct-trees** (voxel grids with heterogeneous resolution: fine cells only near the surface; Tatarchenko et al., 2017) and **nested shape layers** (predict the shape as a composition of positive and negative spaces; Matryoshka Networks).

### Processing voxel inputs: 3D convolution
Exactly like 2D convolution, but the filter is a small 3D cube that slides in three directions. **3D ShapeNets** (Wu et al., 2015) classified $30^3$ voxel shapes with three 3D conv layers. The output-size formula is the same as in 2D (Lecture 7):

In [ ]:
def output_size(W, K, S):
    return (W - K) // S + 1

size = 30
for channels, k, s in [(48, 6, 2), (160, 5, 2), (512, 4, 1)]:
    size = output_size(size, k, s)
    print(f"{k}x{k}x{k} conv, stride {s}: {channels} x {size} x {size} x {size}")
print("then a fully connected layer -> class scores")

In [ ]:
def conv3d(volume, kernel):
    # A 3D convolution (no padding, stride 1): slide a K x K x K kernel through the volume.
    K = kernel.shape[0]
    D = volume.shape[0] - K + 1
    out = np.zeros((D, D, D))
    for i in range(D):
        for j in range(D):
            for k in range(D):
                out[i, j, k] = np.sum(volume[i:i + K, j:j + K, k:k + K] * kernel)
    return out

voxels = voxelize(sphere, 16).astype(float)
edge_detector = -np.ones((3, 3, 3)) / 26
edge_detector[1, 1, 1] = 1.0                              # "me minus the average of my neighbors"
response = conv3d(voxels, edge_detector)
print(f"a 16^3 grid -> {response.shape} after a 3x3x3 convolution; strongest responses are on the sphere's surface:",
      f"{np.mean(response > 0.3):.1%} of the voxels")

### Generating voxel shapes
* **2D CNN → 3D CNN:** encode the image with a 2D CNN, reshape the features into a 3D grid, and upsample with **3D** (transposed) convolutions to a $1\times V\times V\times V$ grid of occupancy scores. Train with a **per-voxel cross-entropy** loss (Choy et al., 3D-R2N2).
* **"Voxel tubes":** cheaper: a 2D CNN whose final conv layer has **$V$ filters**, so each of the $V\times V$ output pixels has $V$ scores, interpreted as a "tube" of voxels along the depth direction. The output $V\times V\times V$ comes from purely 2D operations.

## 3. Implicit functions

**Learn a function to classify arbitrary 3D points as inside or outside the shape:** $o: \mathbb{R}^3 \to [0, 1]$. The surface is the **level set** $\{x : o(x) = \tfrac12\}$ (Occupancy Networks, Mescheder et al., 2019). Same idea: a **signed distance function** (SDF) gives the Euclidean distance to the surface, with the sign telling inside vs. outside.

Because the function can be queried **anywhere**, it gives outputs at **any resolution** (like oct-trees). Let's train a tiny occupancy network in 2D (so we can draw it): an MLP that takes a point $(x, y)$ and outputs P(inside) for a star shape.

In [ ]:
def inside_star(points):
    angle = np.arctan2(points[:, 1], points[:, 0])
    radius = np.linalg.norm(points, axis=1)
    return (radius < 0.5 + 0.25 * np.cos(5 * angle)).astype(int)     # a 5-pointed star

rng = np.random.default_rng(0)
train_points = rng.uniform(-1, 1, (4000, 2))
train_labels = inside_star(train_points)

def init_mlp(sizes, seed=0):
    r = np.random.default_rng(seed)
    return [(r.normal(0, np.sqrt(2 / a), (a, b)), np.zeros(b)) for a, b in zip(sizes[:-1], sizes[1:])]

def mlp_forward(layers, X):
    hs = [X]
    for i, (W, b) in enumerate(layers):
        z = hs[-1] @ W + b
        hs.append(np.maximum(0, z) if i < len(layers) - 1 else z)
    return hs

def mlp_train_step(layers, X, y, lr):
    hs = mlp_forward(layers, X)
    probs = softmax(hs[-1])
    probs[np.arange(len(y)), y] -= 1
    d = probs / len(y)
    for i in reversed(range(len(layers))):                           # backprop, updating as we go
        W, b = layers[i]
        dW, db = hs[i].T @ d, d.sum(axis=0)
        if i > 0:
            d = (d @ W.T) * (hs[i] > 0)
        layers[i] = (W - lr * dW, b - lr * db)

occupancy_net = init_mlp([2, 64, 64, 2])
for step in range(3000):
    batch = rng.integers(0, len(train_points), 256)
    mlp_train_step(occupancy_net, train_points[batch], train_labels[batch], lr=0.1)
test_points = rng.uniform(-1, 1, (5000, 2))
accuracy = np.mean(mlp_forward(occupancy_net, test_points)[-1].argmax(axis=1) == inside_star(test_points))
print(f"occupancy network: inside/outside accuracy on new points {accuracy:.1%}")

Now **query it at any resolution**: the network is a continuous function, not a fixed grid.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 3.3))
for ax, resolution in zip(axes, [16, 64, 256]):
    g = np.linspace(-1, 1, resolution)
    gx, gy = np.meshgrid(g, g)
    grid_points = np.column_stack([gx.ravel(), gy.ravel()])
    p_inside = softmax(mlp_forward(occupancy_net, grid_points)[-1])[:, 1].reshape(resolution, resolution)
    ax.imshow(p_inside > 0.5, origin="lower", cmap="Greys")
    ax.set_title(f"queried on a {resolution}x{resolution} grid"); ax.axis("off")
plt.show()

(−) **Extracting an explicit shape** from an implicit function (e.g., a mesh, for rendering) requires **post-processing**, such as the marching cubes algorithm.

## 4. Point clouds

Represent a shape as a **set of $P$ points** in 3D space.
* (+) Can represent fine structures without huge numbers of points.
* (−) Requires new architectures, losses, etc.
* (−) Doesn't explicitly represent the **surface**: extracting a mesh requires post-processing.

In [ ]:
def sample_shape(kind, n, rng):
    if kind == "sphere":
        p = rng.normal(size=(n, 3))
        return p / np.linalg.norm(p, axis=1, keepdims=True)
    if kind == "cube":                                             # points on the 6 faces of a cube
        p = rng.uniform(-1, 1, (n, 3))
        face_axis = rng.integers(0, 3, n)
        p[np.arange(n), face_axis] = rng.choice([-1.0, 1.0], n)
        return p * 0.8
    if kind == "cylinder":                                         # points on the side of a cylinder
        angle = rng.uniform(0, 2 * np.pi, n)
        return np.column_stack([np.cos(angle), np.sin(angle), rng.uniform(-1, 1, n)])

fig = plt.figure(figsize=(9, 3))
for i, kind in enumerate(["sphere", "cube", "cylinder"]):
    ax = fig.add_subplot(1, 3, i + 1, projection="3d")
    p = sample_shape(kind, 400, rng)
    ax.scatter(p[:, 0], p[:, 1], p[:, 2], s=2)
    ax.set_title(kind); ax.set_axis_off()
plt.show()

### Processing point cloud inputs: PointNet (Qi et al., 2017)
**We want to process point clouds as sets: the order of the points should not matter.** PointNet:
1. Run the **same MLP on each point** independently ($P\times3$ → $P\times D$ point features).
2. **Max-pool** over the points ($D$ numbers): max doesn't care about order.
3. Fully connected layers → class scores.

Let's build it and train it to classify spheres, cubes, and cylinders (each randomly rotated and scaled). As is standard, we first **normalize** each cloud: center it and scale it to fit in the unit sphere.

In [ ]:
def random_rotation(rng):
    q, _ = np.linalg.qr(rng.normal(size=(3, 3)))
    return q

KINDS = ["sphere", "cube", "cylinder"]
def normalize(cloud):
    cloud = cloud - cloud.mean(axis=0)                               # center it
    return cloud / np.linalg.norm(cloud, axis=1).max()               # fit it in the unit sphere

def make_cloud(rng, n_points=128):
    label = int(rng.integers(3))
    cloud = sample_shape(KINDS[label], n_points, rng) @ random_rotation(rng).T * rng.uniform(0.7, 1.3)
    return normalize(cloud), label

def init_pointnet(D=64, seed=0):
    r = np.random.default_rng(seed)
    return {"W1": r.normal(0, np.sqrt(2 / 3), (3, D)), "b1": np.zeros(D),
            "W2": r.normal(0, np.sqrt(2 / D), (D, D)), "b2": np.zeros(D),
            "U": r.normal(0, np.sqrt(1 / D), (D, 3)), "c": np.zeros(3)}

def pointnet_forward(p, cloud):
    z1 = cloud @ p["W1"] + p["b1"]; h1 = np.maximum(0, z1)          # the same MLP on every point
    z2 = h1 @ p["W2"] + p["b2"]; h2 = np.maximum(0, z2)              # (P, D) point features
    winner = h2.argmax(axis=0)                                       # which point won each feature
    pooled = h2.max(axis=0)                                          # max pool over points: (D,)
    return softmax(pooled @ p["U"] + p["c"]), (cloud, z1, h1, z2, h2, winner, pooled)

def pointnet_grads(p, cache, probs, label):
    cloud, z1, h1, z2, h2, winner, pooled = cache
    d_scores = probs.copy(); d_scores[label] -= 1
    g = {"U": np.outer(pooled, d_scores), "c": d_scores}
    d_h2 = np.zeros_like(h2)
    d_h2[winner, np.arange(h2.shape[1])] = p["U"] @ d_scores        # max pooling routes the gradient to the winners
    d_z2 = d_h2 * (z2 > 0)
    g["W2"], g["b2"] = h1.T @ d_z2, d_z2.sum(axis=0)
    d_z1 = (d_z2 @ p["W2"].T) * (z1 > 0)
    g["W1"], g["b1"] = cloud.T @ d_z1, d_z1.sum(axis=0)
    return g

**Order doesn't matter:** shuffle the points, and the output is exactly the same.

In [ ]:
pointnet = init_pointnet()
cloud, _ = make_cloud(rng)
shuffled = cloud[rng.permutation(len(cloud))]
print("PointNet output unchanged when the points are shuffled:", np.allclose(pointnet_forward(pointnet, cloud)[0],
                                                                         pointnet_forward(pointnet, shuffled)[0]))

In [ ]:
state = {k: (np.zeros_like(v), np.zeros_like(v)) for k, v in pointnet.items()}
for t in range(1, 2001):                                             # Adam, minibatches of 16 clouds
    g = {k: np.zeros_like(v) for k, v in pointnet.items()}
    for _ in range(16):
        cloud, label = make_cloud(rng)
        probs, cache = pointnet_forward(pointnet, cloud)
        for name, grad in pointnet_grads(pointnet, cache, probs, label).items():
            g[name] += grad / 16                                     # average the gradient over the batch
    for name in pointnet:
        m, v = state[name]
        m = 0.9 * m + 0.1 * g[name]; v = 0.999 * v + 0.001 * g[name] ** 2
        state[name] = (m, v)
        pointnet[name] -= 0.003 * (m / (1 - 0.9 ** t)) / (np.sqrt(v / (1 - 0.999 ** t)) + 1e-8)

test = [make_cloud(np.random.default_rng(1000 + i)) for i in range(300)]
accuracy = np.mean([pointnet_forward(pointnet, c)[0].argmax() == y for c, y in test])
print(f"PointNet test accuracy on rotated, rescaled shapes: {accuracy:.1%}")

(PointNet++ adds **hierarchy**: it applies PointNet to local neighborhoods, then groups those, like a CNN's growing receptive fields.)

### Generating point clouds
Fan et al. (2017) predict points from an image with two branches: a **fully connected** branch that outputs $P_1\times3$ points, and a **convolutional** branch that outputs $P_2$ points for each of the $H'\times W'$ feature-map positions, for $(P_1 + H'W'P_2)\times3$ points in total.

### Loss function: the Chamfer distance
**We need a (differentiable) way to compare point clouds as sets.** The **Chamfer distance** is the sum of squared L2 distances from each point to its **nearest neighbor in the other set**:

$$d_{CD}(S_1, S_2) = \sum_{x\in S_1}\min_{y\in S_2}\|x - y\|_2^2 + \sum_{y\in S_2}\min_{x\in S_1}\|x - y\|_2^2$$

In [ ]:
def chamfer(A, B):
    squared = ((A[:, None, :] - B[None, :, :]) ** 2).sum(axis=2)         # all pairwise squared distances
    return squared.min(axis=1).sum() + squared.min(axis=0).sum()

A = np.array([[0.0, 0, 0], [1, 0, 0]])
B = np.array([[0.0, 0.1, 0], [1, 0.2, 0], [1, 0, 0.3]])
print(f"Chamfer distance by hand: A->B: 0.01 + 0.04, B->A: 0.01 + 0.04 + 0.09  = {0.01 + 0.04 + 0.01 + 0.04 + 0.09:.2f}")
print(f"computed: {chamfer(A, B):.2f}")

**It's differentiable:** each point gets pulled toward its nearest neighbor in the other set (and toward any points that chose it as *their* nearest neighbor). So we can **fit a point cloud by gradient descent on the point positions**. Start from random points and fit a cube:

In [ ]:
def chamfer_gradient(A, B):
    # Gradient of the Chamfer distance with respect to the points A.
    squared = ((A[:, None, :] - B[None, :, :]) ** 2).sum(axis=2)
    grad = 2 * (A - B[squared.argmin(axis=1)])                        # each a -> its nearest b
    nearest_a = squared.argmin(axis=0)
    np.add.at(grad, nearest_a, 2 * (A[nearest_a] - B))                # each b pulls its nearest a (adds up if shared)
    return grad

target = sample_shape("cube", 500, rng)
points = rng.normal(0, 0.3, (300, 3))                                 # start: a random blob
for step in range(301):
    if step % 100 == 0:
        print(f"step {step}: Chamfer distance {chamfer(points, target):.2f}")
    points -= 0.02 * chamfer_gradient(points, target)

## 5. Triangle meshes

Represent a 3D shape as a set of **triangles**: **vertices** (a set of $V$ points in 3D space) and **faces** (triangles over the vertices).
* (+) The standard representation for graphics; explicitly represents 3D shapes.
* (+) **Adaptive:** flat surfaces need few faces, and more faces can go to areas with fine detail.
* (+) Can attach data to vertices and interpolate over the whole surface (colors, texture coordinates, normals…).
* (−) Nontrivial to process with neural nets!

A simple way to build a sphere mesh: start from an **octahedron** and repeatedly split every triangle into 4, pushing the new vertices out to the sphere:

In [ ]:
def sphere_mesh(subdivisions=2):
    vertices = [np.array(v, float) for v in [(1, 0, 0), (-1, 0, 0), (0, 1, 0), (0, -1, 0), (0, 0, 1), (0, 0, -1)]]
    faces = [(0, 2, 4), (2, 1, 4), (1, 3, 4), (3, 0, 4), (2, 0, 5), (1, 2, 5), (3, 1, 5), (0, 3, 5)]
    for _ in range(subdivisions):
        midpoint_index = {}
        def midpoint(a, b):
            key = (min(a, b), max(a, b))
            if key not in midpoint_index:                          # share midpoints between neighboring triangles
                m = (vertices[a] + vertices[b]) / 2
                vertices.append(m / np.linalg.norm(m))
                midpoint_index[key] = len(vertices) - 1
            return midpoint_index[key]
        new_faces = []
        for a, b, c in faces:                                      # split each triangle into 4
            ab, bc, ca = midpoint(a, b), midpoint(b, c), midpoint(c, a)
            new_faces += [(a, ab, ca), (b, bc, ab), (c, ca, bc), (ab, bc, ca)]
        faces = new_faces
    return np.array(vertices), np.array(faces)

vertices, faces = sphere_mesh(2)
edges = sorted({(min(f[i], f[(i + 1) % 3]), max(f[i], f[(i + 1) % 3])) for f in faces for i in range(3)})
print(f"sphere mesh: {len(vertices)} vertices, {len(faces)} faces, {len(edges)} edges")

### Predicting meshes: Pixel2Mesh (Wang et al., 2018)
Input: a single RGB image. Output: a triangle mesh. Four key ideas:

**Idea #1: iterative refinement.** Start from an initial ellipsoid mesh; the network predicts an **offset** for each vertex; repeat.

**Idea #2: graph convolution.** The input is a graph with a feature vector at each vertex; the output is a new feature vector for each vertex. Vertex $i$'s new feature depends on its own feature and its **neighbors'**, with the same weights $W_0$, $W_1$ for every vertex:

$$f'_i = W_0f_i + \sum_{j\in N(i)}W_1f_j$$

In [ ]:
neighbors = [[] for _ in range(len(vertices))]
for a, b in edges:
    neighbors[a].append(b); neighbors[b].append(a)

def graph_conv(features, W0, W1):
    out = features @ W0                                               # each vertex's own feature
    for i, nbrs in enumerate(neighbors):
        out[i] += features[nbrs].sum(axis=0) @ W1                     # plus its neighbors'
    return out

features = vertices.copy()                                            # start with each vertex's position as its feature
W0, W1 = rng.normal(size=(3, 8)), rng.normal(size=(3, 8)) * 0.2
print("graph convolution:", features.shape, "->", graph_conv(features, W0, W1).shape,
      f" (vertices have {min(map(len, neighbors))}-{max(map(len, neighbors))} neighbors)")

**Idea #3: vertex-aligned features.** How do we use the image? For each vertex, **project it onto the image plane** with the camera, and use **bilinear interpolation** to sample the CNN feature there (like RoI Align from Lecture 15: it keeps the features aligned with the input image):

In [ ]:
def bilinear(feature_map, x, y):
    x0, y0 = int(np.floor(x)), int(np.floor(y))
    dx, dy = x - x0, y - y0
    return (feature_map[y0, x0] * (1 - dx) * (1 - dy) + feature_map[y0, x0 + 1] * dx * (1 - dy)
            + feature_map[y0 + 1, x0] * (1 - dx) * dy + feature_map[y0 + 1, x0 + 1] * dx * dy)

feature_map = rng.normal(size=(20, 20, 16))                           # a CNN feature map: 20 x 20, 16 channels
camera_vertex = np.array([0.3, -0.2, 3.0])                            # a mesh vertex in front of the camera
u, v = 10 + 8 * camera_vertex[0] / camera_vertex[2], 10 + 8 * camera_vertex[1] / camera_vertex[2]   # project
print(f"vertex projects to feature-map position ({u:.2f}, {v:.2f}); sampled feature shape {bilinear(feature_map, u, v).shape}")

**Idea #4: the loss function.** **The same shape can be represented with different meshes**, so how do we compare a predicted and a ground-truth mesh? **Convert both to point clouds, then use the Chamfer distance:**
* sample points from the **ground-truth** mesh's surface (offline);
* sample points from the **predicted** mesh's surface **online** (GEOMetrics, Smith et al., 2019). Using only the predicted **vertices** would ignore the interior of the faces.

Sampling online must be efficient, and **we must backprop through the sampling**. The trick: sample a triangle (with probability proportional to its area) and a random point in it, written as a weighted average of the triangle's corners, $p = w_1v_a + w_2v_b + w_3v_c$ (barycentric coordinates). Then the gradient flows back to the corners: $\frac{\partial L}{\partial v_a} = w_1\frac{\partial L}{\partial p}$, and so on.

In [ ]:
def sample_surface(vertices, faces, n, rng):
    a, b, c = vertices[faces[:, 0]], vertices[faces[:, 1]], vertices[faces[:, 2]]
    areas = 0.5 * np.linalg.norm(np.cross(b - a, c - a), axis=1)
    chosen = rng.choice(len(faces), n, p=areas / areas.sum())        # bigger triangles get more samples
    r1, r2 = np.sqrt(rng.random(n)), rng.random(n)
    weights = np.column_stack([1 - r1, r1 * (1 - r2), r1 * r2])      # uniform random barycentric coordinates
    corners = vertices[faces[chosen]]                                # (n, 3 corners, 3 coordinates)
    points = np.sum(weights[:, :, None] * corners, axis=1)
    return points, chosen, weights

def backprop_to_vertices(grad_points, faces, chosen, weights, n_vertices):
    grad_vertices = np.zeros((n_vertices, 3))
    for corner in range(3):                                          # each corner gets w * dL/dp
        np.add.at(grad_vertices, faces[chosen, corner], weights[:, corner:corner + 1] * grad_points)
    return grad_vertices

### Fitting a mesh, and why we need a regularizer
Deform the sphere mesh to fit a **cube** by gradient descent on its vertex positions (iterative refinement, without a network). **Using the Chamfer distance as the only loss gives degenerate meshes**, so Mesh R-CNN adds a **mesh regularizer**: $L_{edge}$ = the squared length of every edge, which keeps the triangles small and even. Compare:

In [ ]:
def fit_mesh(edge_weight, steps=300, lr=2.0, n_samples=500, seed=0):
    r = np.random.default_rng(seed)
    verts, mesh_faces = sphere_mesh(3)
    edge_list = np.array(sorted({(min(f[i], f[(i + 1) % 3]), max(f[i], f[(i + 1) % 3])) for f in mesh_faces for i in range(3)}))
    target = sample_shape("cube", 1000, r)
    for step in range(steps):
        points, chosen, weights = sample_surface(verts, mesh_faces, n_samples, r)       # sample the PREDICTED mesh
        grad = backprop_to_vertices(chamfer_gradient(points, target) / n_samples, mesh_faces, chosen, weights, len(verts))
        edge_vectors = verts[edge_list[:, 0]] - verts[edge_list[:, 1]]                # gradient of the average squared edge length
        np.add.at(grad, edge_list[:, 0], edge_weight * 2 * edge_vectors / len(edge_list))
        np.add.at(grad, edge_list[:, 1], -edge_weight * 2 * edge_vectors / len(edge_list))
        verts = verts - lr * grad
    final_points, _, _ = sample_surface(verts, mesh_faces, 2000, r)
    lengths = np.linalg.norm(verts[edge_list[:, 0]] - verts[edge_list[:, 1]], axis=1)
    return verts, mesh_faces, chamfer(final_points, target) / 3000, lengths

fits = {}
for edge_weight in [0.0, 1.0]:
    verts, mesh_faces, cd, lengths = fit_mesh(edge_weight)
    fits[edge_weight] = (verts, mesh_faces)
    print(f"edge regularizer weight {edge_weight}: Chamfer {cd:.4f}, edge lengths: spread (std/mean) {lengths.std() / lengths.mean():.2f}, "
          f"longest edge = {lengths.max() / lengths.mean():.1f} x the average")

In [ ]:
fig = plt.figure(figsize=(9, 4))
for i, (edge_weight, (verts, mesh_faces)) in enumerate(fits.items()):
    ax = fig.add_subplot(1, 2, i + 1, projection="3d")
    ax.plot_trisurf(verts[:, 0], verts[:, 1], verts[:, 2], triangles=mesh_faces, color="lightsteelblue", edgecolor="gray", linewidth=0.2)
    ax.set_title(f"edge regularizer weight {edge_weight}"); ax.set_axis_off()
plt.show()

**What we see:** both meshes fit the cube about equally well (similar Chamfer distance), but without the regularizer the triangles become **uneven**: some edges get stretched to about 2.5× the average length (long slivers, bad for rendering and for further refinement). The edge regularizer keeps the edges much more uniform at a small cost in Chamfer distance.

## 6. Shape comparison metrics

**Intersection over union (IoU)** works for 2D boxes and masks, and in 3D as **voxel IoU**. But: it can't capture **thin structures**, can't be applied to point clouds directly, needs voxelizing or sampling for meshes, and **isn't very meaningful at low values** (state-of-the-art methods only reach about 0.5–0.57 on ShapeNet). *Conclusion: voxel IoU is not a good metric.*

**Chamfer distance:** convert the prediction and ground truth to point clouds by sampling, and compare. **Problem: Chamfer is very sensitive to outliers** (it squares distances).

**F1 score:** sample points from both surfaces, then, for a threshold $t$:
* **precision@t** = fraction of **predicted** points within $t$ of some ground-truth point;
* **recall@t** = fraction of **ground-truth** points within $t$ of some predicted point;
* $\text{F1@}t = \frac{2\cdot\text{precision}\cdot\text{recall}}{\text{precision} + \text{recall}}$.

The slides' example: precision 3/4 and recall 2/3:

In [ ]:
def f1_score(pred, truth, t):
    distances = np.sqrt(((pred[:, None, :] - truth[None, :, :]) ** 2).sum(axis=2))
    precision = np.mean(distances.min(axis=1) < t)          # predicted points near the truth
    recall = np.mean(distances.min(axis=0) < t)             # true points covered by the prediction
    return 2 * precision * recall / (precision + recall) if precision + recall else 0.0, precision, recall

truth = np.array([[0.0, 0, 0], [1, 0, 0], [2, 0, 0]])
pred = np.array([[0.05, 0, 0], [0.0, 0.05, 0], [1.05, 0, 0], [5, 5, 5]])   # covers 2 of 3 true points; 1 outlier
f1, p, r = f1_score(pred, truth, t=0.1)
print(f"precision {p:.2f} (= 3/4), recall {r:.2f} (= 2/3), F1 {f1:.2f}  (slides: about 0.70)")

**F1 is robust to outliers, Chamfer isn't.** Take a good prediction and move just **one** of its 500 points far away:

In [ ]:
truth_cloud = sample_shape("sphere", 500, rng)
good = sample_shape("sphere", 500, rng)
with_outlier = good.copy()
with_outlier[0] = [10.0, 10.0, 10.0]                          # one bad point out of 500
for name, cloud in [("good prediction", good), ("same, plus 1 outlier", with_outlier)]:
    print(f"{name:22s}: Chamfer {chamfer(cloud, truth_cloud):8.2f},  F1@0.2 {f1_score(cloud, truth_cloud, 0.2)[0]:.3f}")

One outlier explodes the Chamfer distance, while F1 barely moves. *Conclusion: F1 is probably the best shape prediction metric in common use* (looking at several thresholds $t$ to capture details at different scales).

| Metric | Pros | Cons |
|---|---|---|
| Voxel IoU | familiar | misses fine structure, not meaningful at low values |
| Chamfer | can be directly optimized | very sensitive to outliers |
| F1@t | robust to outliers | need several thresholds for different scales |

## 7. Cameras, datasets, and Mesh R-CNN

### Canonical vs. view coordinates
* **Canonical coordinates:** predict the shape in a fixed coordinate system for each category (e.g., the front of a chair is $+z$), regardless of the viewpoint. Many papers do this because the data is easier to load.
* **View coordinates:** predict the shape aligned to the **camera's** viewpoint.

**Problems with canonical coordinates:** they break the "**principle of feature alignment**" (predictions should be aligned to the inputs), and they **overfit** to the training shapes: better generalization to new views of known shapes, but worse to new shapes or new categories (Shin et al., 2018: voxel IoU 0.902 vs. 0.714 on novel views, but 0.474 vs. 0.517 on novel models and 0.309 vs. 0.570 on novel categories, canonical vs. view). *Conclusion: prefer view coordinates.* (In view coordinates, voxels follow the perspective camera, so they're actually **frustums**.)

### 3D datasets
* **ShapeNet:** about 50 categories, about 50,000 3D CAD models; the standard split has 13 categories, about 44,000 models, 25 rendered images each. (−) Synthetic, isolated objects with no context; lots of chairs, cars, and airplanes.
* **Pix3D:** 9 categories, 219 3D models of IKEA furniture aligned to about 17,000 **real** images. (+) Real images with context. (−) Small, with partial annotations (only one object per image).

### Mesh R-CNN (Gkioxari, Malik & Johnson, 2019)
**Task:** input a single RGB image; output a set of detected objects, each with a box, a category, an instance mask, **and a 3D triangle mesh**: Mask R-CNN plus a **mesh head**.

**A hybrid 3D shape representation:** mesh deformation (like Pixel2Mesh) gives good results, but the **topology** (number of holes, connected components) is fixed by the initial sphere. A chair with holes can't be made from a sphere by moving vertices! **Mesh R-CNN's approach: use a voxel prediction to create the initial mesh**, then refine it:
1. 2D object recognition (Mask R-CNN);
2. predict coarse **3D voxels** for each object;
3. convert the voxels to a mesh ("**cubify**": each occupied voxel becomes a cube, shared faces removed);
4. **refine the mesh** with graph convolutions and vertex-aligned features, using Chamfer loss plus the **edge-length regularizer**.

On Pix3D it predicts many objects per scene and even completes **occluded** parts of objects ("amodal completion"); when the 2D segmentation fails, the errors propagate to the meshes.

## 8. In practice (optional)
**PyTorch3D** (from the same group) implements meshes, graph convolutions, differentiable rendering, and the Chamfer loss on the GPU:

In [ ]:
# [PyTorch] Needs: pip install torch pytorch3d
import torch
from pytorch3d.utils import ico_sphere
from pytorch3d.ops import sample_points_from_meshes
from pytorch3d.loss import chamfer_distance, mesh_edge_loss

mesh = ico_sphere(level=2)                                        # an initial sphere mesh
points = sample_points_from_meshes(mesh, num_samples=1000)        # differentiable surface sampling
target = torch.randn(1, 1000, 3)
loss, _ = chamfer_distance(points, target)
print("Chamfer:", loss.item(), " edge loss:", mesh_edge_loss(mesh).item())

## Summary
| Representation | Process with | Predict with | Pros | Cons |
|---|---|---|---|---|
| **Depth map** | 2D CNN | fully convolutional net, scale-invariant loss | simple, matches sensors | only the visible surface; scale ambiguity |
| **Voxels** | 3D CNN | 2D→3D CNN or voxel tubes, per-voxel cross-entropy | simple grid | memory grows as $V^3$ |
| **Implicit** | MLP on points | occupancy / SDF networks | any resolution | needs post-processing to extract a mesh |
| **Point cloud** | PointNet (shared MLP + max pool) | point generation + Chamfer loss | fine detail, cheap | no explicit surface |
| **Mesh** | graph convolution | iterative refinement + vertex-aligned features + sampled Chamfer + regularizer | standard in graphics, adaptive | hard for neural nets; fixed topology |

**Metrics:** prefer **F1@t** over voxel IoU and Chamfer. **Coordinates:** prefer view coordinates. **Mesh R-CNN** = Mask R-CNN + voxels → cubify → mesh refinement.

### Check your understanding
1. Why can't a single image tell you the absolute depth of an object? How does the scale-invariant loss handle it?
2. How much memory does a $512^3$ grid of 32-bit voxels need?
3. Why is PointNet's output the same if you shuffle the input points?
4. Why does Pixel2Mesh sample points from the *predicted* mesh's faces, not just use its vertices?
5. One stray point makes your Chamfer distance huge. Which metric should you report instead, and why?
6. Why can't deforming a sphere mesh produce a chair with holes, and how does Mesh R-CNN fix that?

<details><summary>Answers</summary>

1. A small close object and a large far one project identically ($f\cdot X/Z$). The loss subtracts the average log-difference, so a global scale factor costs nothing.
2. $4 \times 512^3$ bytes = 512 MB.
3. The same MLP is applied to each point independently, and max pooling doesn't depend on the order.
4. Vertices alone ignore the interior of the faces; a mesh could have its vertices on the target surface while its faces cut through empty space.
5. F1@t: it counts the fraction of points within a threshold, so one outlier only changes it by one point's worth.
6. Moving vertices can't change the topology (number of holes) of the initial mesh. Mesh R-CNN first predicts voxels, which can have any topology, converts them to a mesh, and then refines it.
</details>

**Next:** Lecture 18, videos.